# 기준(EXP-011) 오차 분석: 정답을 상위 5위에 넣지 못하는 질의

dev-full summary_q에서 기준 순위가 정답 문서를 상위 5위 밖에 둔 질의를 맞힌 질의와 비교해, 실패 유형과 보완 방향을 찾는다. 라벨 필드(`question_comment`·`answer_comment`, 정답 문서의 원래 질문·답변 발언)는 분석에만 쓰고 검색 입력에는 쓰지 않는다. EXP 판정이 아니다.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from kiwipiepy import Kiwi

from rag.bm25 import forms
from rag.conditions import build_extractor
from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever, chunks_by_doc
from rag.search import query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
BASE_RUN = ROOT / 'data/runs/snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-full.csv'
TOP = 5
NEAR = 20
DEICTIC = r'(^|\s)(그|이|저|해당|이러한|그러한|이런|그런|위|앞서)\s'
N_SAMPLE = 12
SNIP = 220
SEED = 20260929

## 실패 질의

In [3]:
cfg = load_cfg(['+exp=exp011', 'index.scope=full'])
gold = load_gold(cfg, 'dev-full')
run = read_run(BASE_RUN)
rank = gold_ranks(run, gold, 10)
miss = ~(rank <= TOP)
print(len(gold), 'queries, 실패', int(miss.sum()), f'({miss.mean():.1%})')
print('정답 6~10위', int(((rank > TOP) & (rank <= 10)).sum()), ' 10위 밖', int(rank.isna().sum()))

3016 queries, 실패 200 (6.6%)
정답 6~10위 71  10위 밖 129


In [4]:
labels = {}
with (ROOT / cfg.paths.queries).open(encoding='utf-8') as f:
    for line in f:
        q = json.loads(line)
        if q['qid'] in gold.index:
            labels[q['qid']] = q
docs = {}
need = set(gold['doc_id'])
with (ROOT / cfg.paths.corpus).open(encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        docs[d['doc_id']] = {k: d[k] for k in ['context', 'conference_number', 'meeting_name',
                                              'committee_name', 'agenda', 'n_qa']}
print(len(labels), 'labels,', len(docs), 'docs')

3016 labels, 38516 docs


## 목록별 정답 순위

결합 전 세 목록(dense, BM25, 조건)에서 정답이 몇 위였는지로 실패를 나눈다.

In [5]:
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = dict(zip(qids, read_texts(text_path, qids), strict=True))
q_all = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
q_emb = dict(zip(qids, q_all, strict=True))
ex = build_extractor(cfg)
tok = dict(zip(qids, ret.query_tokens([texts[q] for q in qids]), strict=True))
doc_chunks = chunks_by_doc(ret.dense_ids)
rows = []
for q in qids:
    row = ret.chunk_emb @ q_emb[q]
    named, matched, _ = ret.lists(row, tok[q], ex.extract(texts[q]))
    g = gold.loc[q, 'doc_id']
    pos = {n: (lst.index(g) + 1 if g in lst else np.nan) for n, lst in named.items()}
    top1 = run[(run['qid'] == q) & (run['rank'] == 1)]['doc_id'].iloc[0]
    ranks = {f'{n}_rank': pos.get(n, np.nan) for n in ['dense', 'bm25', 'conditions']}
    rows.append({'qid': q, **ranks,
                 'has_cond': 'conditions' in named, 'gold_best_match': g in matched,
                 'n_best_match': len(matched), 'top1': top1,
                 'gold_dense': float(row[doc_chunks[g]].max()),
                 'top1_dense': float(row[doc_chunks[top1]].max())})
lr = pd.DataFrame(rows).set_index('qid')
lr['final'] = rank
lr['miss'] = miss
print(lr[lr['miss']].describe().round(3).to_string())

       dense_rank  bm25_rank  conditions_rank  n_best_match  gold_dense  top1_dense   final
count     129.000    143.000              0.0       200.000     200.000     200.000  71.000
mean       28.775     21.231              NaN         0.030       0.470       0.558   7.831
std        27.806     23.445              NaN         0.424       0.105       0.074   1.434
min         1.000      1.000              NaN         0.000       0.085       0.269   6.000
25%         7.000      2.000              NaN         0.000       0.404       0.512   6.500
50%        18.000     13.000              NaN         0.000       0.475       0.557   8.000
75%        46.000     31.000              NaN         0.000       0.532       0.603   9.000
max       100.000     92.000              NaN         6.000       0.797       0.838  10.000


In [6]:
best = lr[['dense_rank', 'bm25_rank', 'conditions_rank']].min(axis=1)
cat = pd.Series('전 목록 실패(>' + str(NEAR) + ')', index=lr.index)
cat[best <= NEAR] = f'근접({TOP + 1}~{NEAR})'
cat[best <= TOP] = '결합 손실(한 목록은 top5)'
lr['cat'] = cat
m = lr[lr['miss']]
print(m['cat'].value_counts().to_string())
print('\n실패 중 목록별 top5:', {n: int((m[f'{n}_rank'] <= TOP).sum())
                              for n in ['dense', 'bm25', 'conditions']})

cat
결합 손실(한 목록은 top5)    81
전 목록 실패(>20)         69
근접(6~20)             50

실패 중 목록별 top5: {'dense': 28, 'bm25': 54, 'conditions': 0}


## 같은 회의의 다른 문서와의 혼동

In [7]:
def conf(d):
    return docs[d]['conference_number']


def sibling(d, q):
    g = gold.loc[q, 'doc_id']
    return d != g and conf(d) == conf(g)


lr['top1_same_conf'] = [sibling(r.top1, q) for q, r in lr.iterrows()]
top5 = run[run['rank'] <= TOP].groupby('qid')['doc_id'].apply(list)
lr['top5_same_conf'] = [sum(sibling(d, q) for d in top5[q]) for q in lr.index]
conf_size = pd.Series([d['conference_number'] for d in docs.values()]).value_counts()
lr['conf_docs'] = [conf_size[docs[gold.loc[q, 'doc_id']]['conference_number']] for q in lr.index]
print(lr.groupby('miss')[['top1_same_conf', 'top5_same_conf', 'conf_docs']].mean().round(3))

       top1_same_conf  top5_same_conf  conf_docs
miss                                            
False           0.021           0.356     12.605
True            0.105           0.300     15.005


## 질의 특성

In [8]:
lr['q_len'] = [len(texts[q]) for q in lr.index]
lr['deictic'] = [bool(re.search(DEICTIC, texts[q])) for q in lr.index]
lr['qna_type'] = gold['qna_type']
lr['meeting'] = gold['meeting_name']
print(lr.groupby('miss')[['q_len', 'deictic', 'has_cond']].mean().round(3))
for col in ['qna_type', 'meeting']:
    t = lr.groupby(col)['miss'].agg(['size', 'mean']).round(3)
    print('\n', t.rename(columns={'size': 'n', 'mean': '실패율'}).to_string())

        q_len  deictic  has_cond
miss                            
False  71.621    0.152     0.001
True   51.545    0.145     0.005

              n    실패율
qna_type             
단답형        289  0.100
추출형       2727  0.063

               n    실패율
meeting               
국정감사       1225  0.047
본회의         112  0.089
소위원회       1214  0.070
예산결산특별위원회   158  0.108
특별위원회       307  0.101


## 정답 문서 특성과 근거 위치

In [9]:
def locate(q):
    ctx = docs[gold.loc[q, 'doc_id']]['context']
    span = labels[q]['answer_comment'].strip()[:30]
    i = ctx.find(span)
    return i / len(ctx) if i >= 0 else np.nan


lr['doc_chars'] = [len(docs[gold.loc[q, 'doc_id']]['context']) for q in lr.index]
lr['n_chunks'] = [len(doc_chunks[gold.loc[q, 'doc_id']]) for q in lr.index]
lr['answer_pos'] = [locate(q) for q in lr.index]
lr['n_qa'] = [int(docs[gold.loc[q, 'doc_id']]['n_qa']) for q in lr.index]
print(lr.groupby('miss')[['doc_chars', 'n_chunks', 'n_qa', 'answer_pos']].median().round(3))
print('근거 발언을 문서에서 찾은 비율', lr['answer_pos'].notna().mean().round(3))

       doc_chars  n_chunks  n_qa  answer_pos
miss                                        
False     1894.5       7.0   1.0       0.528
True      1611.0       6.0   1.0       0.590
근거 발언을 문서에서 찾은 비율 0.996


## 어휘 겹침

질의 형태소(BM25와 같은 품사) 중 정답 문서와 1위 문서에 나오는 비율. 실패 질의에서 1위 문서 쪽이 높으면 질의어가 정답보다 다른 문서를 더 잘 설명한다는 뜻이다.

In [10]:
kiwi = Kiwi()
tags = set(cfg.retriever.bm25.tags)
m_q = lr[lr['miss']].index.tolist()
q_forms = dict(zip(m_q, map(set, forms(kiwi, [texts[q] for q in m_q], tags)), strict=True))
need_docs = {gold.loc[q, 'doc_id'] for q in m_q} | set(lr.loc[m_q, 'top1'])
d_forms = dict(zip(need_docs, map(set, forms(kiwi, [docs[d]['context'] for d in need_docs], tags)),
                   strict=True))
def share(q, d):
    return len(q_forms[q] & d_forms[d]) / max(1, len(q_forms[q]))


ov = pd.DataFrame({q: {'gold': share(q, gold.loc[q, 'doc_id']), 'top1': share(q, lr.loc[q, 'top1'])}
                   for q in m_q}).T
print(ov.describe().round(3).to_string())
print('1위 문서 겹침이 더 큼', int((ov['top1'] > ov['gold']).sum()), '/', len(ov))

          gold     top1
count  200.000  200.000
mean     0.634    0.741
std      0.224    0.165
min      0.000    0.280
25%      0.500    0.624
50%      0.667    0.750
75%      0.800    0.875
max      1.000    1.000
1위 문서 겹침이 더 큼 118 / 200


## 실패 사례 (무작위 N_SAMPLE개)

질의, 정답 문서의 원래 질문 발언(앞부분), 정답 문서와 1위 문서의 회의·안건. 유형을 손으로 분류하는 근거다.

In [11]:
rng = np.random.default_rng(SEED)
sample = sorted(rng.choice(m_q, size=N_SAMPLE, replace=False))
for q in sample:
    g, t = gold.loc[q, 'doc_id'], lr.loc[q, 'top1']
    r = lr.loc[q]
    print(f'[{q}] {r["cat"]}, 최종 {r["final"]}, 1위 같은 회의 {r["top1_same_conf"]}')
    print('  질의  ', texts[q].strip())
    print('  원 질문', labels[q]['question_comment'].strip()[:SNIP])
    print('  정답  ', docs[g]['committee_name'], '|', docs[g]['agenda'][:80])
    print('  1위   ', docs[t]['committee_name'], '|', docs[t]['agenda'][:80])
    print()

[032448-0001] 결합 손실(한 목록은 top5), 최종 7.0, 1위 같은 회의 False
  질의   소득불평등 증가와 서민 및 중산층의 어려움에 대응하기 위한 구체적 대책이 어떻게 진행되고 있으며, 추가적인 대책은 어떻게 계획하고 계십니까?
  원 질문 십분위 배율로 본 소득불평등지수는 2004년 9.08배로 IMF 당시 수준으로 돌아갔고 저소득층의 적자율도 99년 15.4%에서 2004년 18.1%를 기록했습니다.  지금 서민대책과 중산층의 삶이 붕괴되고 있습니다. 거시적 대책 외에 실천적ㆍ구체적 대책이 광범위하게 필요한데 대책을 가지고 계신 것이 있습니까?
  정답   국회본회의 | 2. 경제에관한질문
  1위    국회본회의 | 1. 경제에 관한 질문(계속)

[046414-0020] 근접(6~20), 최종 6.0, 1위 같은 회의 False
  질의   현재의 저출산 문제를 해결하기 위해 새로운 방안을 모색하는 과정에서, 어떤 정책적 개입이 효과적일지를 결정하는 데 있어 어떤 요소들이 주요한가요?
  원 질문 덧붙여서 말씀드리면 아동수당세는 단순히 아동수당을 위한 재원 마련 방안만은 아닙니다. 우리 사회 초고소득자와 경제를 책임지고 있는 거대 법인들에게 사회적 역할을 요청하는 것입니다.  지금 우리가 이 저출산 문제를 해소하지 못하면 나라의 장래가 정말 위태로울 지경인데 대한민국 공동체가 함께 힘을 모으자는 것이지요. 그래서 아이들을 키우자는 것입니다. 그런 취지를 갖고 있습니다.  동의하십니까,
  정답   국회본회의 | 1. 경제에 관한 질문
  1위    저출산·고령화대책특별위원회 | 1. 업무보고가. 기획재정부나. 행정자치부다. 고용노동부

[047529-0007] 결합 손실(한 목록은 top5), 최종 nan, 1위 같은 회의 False
  질의   인건비 부족 문제에 대처하기 위해 월급을 줄이는 방안을 검토할 생각이 있으신가요?
  원 질문 인건비가 지금 부족한 부분이오. 그러면 월급 안 줄 겁니까?
  정답   예산결산특별위원회

## 정리

- 실패 200개(6.6%): 정답 6~10위 71, 10위 밖 129
- 목록별: 결합 손실 81(40%, 한 목록은 정답을 top5에 둠. BM25 top5 54, dense top5 28), 근접(최선 목록 6~20위) 50, 전 목록 실패(>20위) 69. 조건 목록은 summary_q에 거의 생기지 않는다(0.5%)
- 같은 회의의 다른 문서가 1위: 실패 10.5% vs 성공 2.1%(실패 중 21개)
- 질의: 실패 질의가 짧다(중앙 52자 vs 72자). 지시어 비율은 같다(14.5% vs 15.2%). 단답형 실패율 10.0% vs 추출형 6.3%, 예결위 10.8%·특별위 10.1% vs 국정감사 4.7%
- 어휘: 실패 질의의 형태소는 정답 문서보다 1위 문서에 더 많이 나온다(중앙 0.75 vs 0.67, 200개 중 118개). 질의어가 정답에 없어서가 아니라 다른 문서를 더 잘 설명해서 진다
- 사례 12개(손 분류): 주제가 일반적이라 관련 문서가 많음 5(소득불평등, 저출산, 벤처, 대북제재, 종부세. 1위 문서도 그럴듯함), 질의를 만들며 원 발언의 구분 단서(위원 이름, 사건, 핵심어)가 빠짐 3, 지시 대상이 질의에 없음 3("이 사업", "이 세율"), 원 발언과 다른 말로 바뀜 1
- 보완 방향: (1) 결합 손실 40%는 상위 후보를 다시 줄 세우는 쪽(리랭커, RRF k 등 결합 방식)이 겨냥하는 몫이다. (2) 질의 확장(재작성·HyDE)은 질의어를 더하는 방식이라 "다른 문서를 더 잘 설명함" 유형에는 맞지 않는다. (3) 일반 주제·단서 손실은 정답 하나 라벨의 한계에 가깝고 검색으로 고치기 어렵다